In [1]:
!pip install openai pandas pymupdf

In [2]:
import pandas as pd
import json
import os
from openai import OpenAI
import fitz
import base64
from datetime import datetime
import getpass

In [3]:
api_key = getpass.getpass("Enter your OpenAI API key: ")
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
pdf_path = input("Enter the full path to your PDF file: ").strip().strip('"')
if not os.path.exists(pdf_path): 
    print("File not found. Please check the path and try again.")
else:
    print("File found")

Enter your OpenAI API key:  ········
Enter the full path to your PDF file:  "C:\Users\tdmur\Downloads\The Scientific World Journal - 2025 - Addis - Intestinal Parasitic Infection  Prevalence and Associated Risk Factors at.pdf"


File found


In [5]:
outputdirectory = "data/outputs"
os.makedirs(outputdirectory, exist_ok = True)
#log entries is a list that populates with the information of each table that is scanned from the pdf. It is crucial because
#it contains information about each table (what issues were flagged, what source are these tables from, when was this extracted, etc.
#table_count counts how many tables are scanned.
log_entries = []
table_count = 0

#fitz is a package that facilitates 
pdf_document = fitz.open(pdf_path)
total_pages = len(pdf_document)

#outputs a message telling the user that the agent has successfully opened the pdf, and lets the reader know how many pages
# it detects.
print(f"Opened PDF: {pdf_path}")
print(f"Total Pages: {total_pages}")

#Establishes loop through every page that denotes how many pages scanned, transforms the table images in the PDFs to
#readable images for GPT Vision and creates a clean, workable, and readable response.
for page_num in range(len(pdf_document)):
    page = pdf_document[page_num]
#Converts the page to a readable image for OpenAI API "GPT Vision"    
    mat = fitz.Matrix(2, 2)
    pix = page.get_pixmap(matrix=mat)
    img_bytes = pix.tobytes("png")
    img_base64 = base64.b64encode(img_bytes).decode("utf-8")
    
#Builds a prompt that asks ChatGPT if there are tables in the image.This is done as a variable so its less confusing to look at
#compared to the alternative of putting between a plethora of brackets and parenthesis.
    extraction_prompt = """ You are a highly skilled tool used for data extraction from PDF files.
Carefully examine this PDF page and extract ALL tables exactly as they appear. 
For each table: 
- Record the table title if visible on the page
- Preserve exact column headers including any units
- Do not merge, split, or summarize ANY cells
- If a cell appears empty, represent it as an empty string ""
- Flag any cells that appear merged, shifted, or ambiguous
- If a table has subheaders or merged header rows, flatten them into a single header row by combining the parent and child header with a dash. 
- Every row must have exactly the same number of values as there are headers
- Count your headers carefully before writing rows — the number of values in EVERY row must equal the number of headers exactly
- If you are unsure about a merged cell, duplicate the parent label for each child column rather than leaving it empty
- If a page contains multiple tables, extract them one at a time and include each as a separate entry in the tables array
- Focus on accuracy over speed

When validating your work,
- Check every column for missing or empty valuyes
- Check that row labels are present and not shifted (in the correct place according to the original material)
- Check that the number of values in each row matches the number of headers
- Check that the number of values in each column matches number of columns
- Give a verdict of Approved if the table looks clean and complete
- Give a verdict of Requires Correction if any issues were found
- Based on what you know as A highly skilled tool used for data extraction, make a suggestion in the table about how the user can fix tables that "Need Correction" output it in the suggestions part.
- List issues and suggestions where necessary

Respond ONLY in JSON with no extra text or markdown"
{"tables": [{"title": "table title or empty string", "headers": ["column1", "column2"], "rows": [["value1", "value2"]], "Verdict": "Approved", "Issues": [], "Suggestion": ""}]}
If no tables were found on this page, respond with: {"tables":[]}"""

#This section of code is when the prompt is sent to GPT. It outlines GPT's role and what it should expect inputs should be.
#the two "content" lines tell GPT that an image is included and the GPT must use GPT vision to carry out the instructions
#given to it by the prompt. 
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        max_tokens=8000,
        messages=[{
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/png;base64,{img_base64}"
                    }
                },
                { 
                    "type": "text",
                    "text": extraction_prompt
                }
            ]
        }]
    )
#This code goes through each JSON formatted GPT response and ensures that a proper, readable JSON structure is in the output. 
#the "Try , except" works as a fail safe to essentially ignore and move on from the strict JSON instructions for a table if there
# is an error in outputting it in such a way. Thus, the code doesn't crash with a faulty, less clean GPT response.
    clean = response.choices[0].message.content.replace("```json", "").replace("```", "").strip()
    print(f"\nProcessing page {page_num+1} of {total_pages}...")
    try:
        parsed = json.loads(clean)
    except json.JSONDecodeError:
        print(f"Could not parse response on page {page_num + 1}. Skipping")
        continue
#This code creates a dictionary entry for each of the tables that was gathered by GPT and pairs it with the page it was on. 
#It disregards the pages shown to not have tables. It then prints how many tables were found on the pages that included tables and prints their titles.
    tables = parsed.get("tables", [])
    if not tables:
        print(f"No tables found on page {page_num + 1}.")
        continue
    else:
        print(f"Tables found: {len(tables)}")
        for table in tables:
            print(f"{table.get("title", "Untitled table")}")
#This for loop assigns each table in the dictionary of tables to a table ID to be referenced and printed. To ensure each table ID includes the proper information,
#a .get function is used to take that information from the dictionary list and assigns them.
    for table_id, table in enumerate(tables):
        headers = table.get("headers", [])
        rows = table.get("rows", [])
        title = table.get("title", "").strip()
        verdict = table.get("Verdict", "Unknown")
        issues = table.get("Issues", [])
        suggestion = table.get("Suggestion", "")
#This acts as a fail safe, telling the agent to skip tables where it can not decipher if the table has headers or rows.
#It prevents the entire code from crashing because of one bad table. 
        if not headers or not rows:
            print(f"No headers or rows. Skipping.")
            continue
            
#This is a filter tool. It is useful for identifying tables that are incomplete or have missing data that should be reviewed.
# Each row should correspond to a header, and if it doesn't it likely means data is missing or the table is incomplete.
#Additionally, a log entry is created each time this happens. 
        row_lengths = [len(row) for row in rows]
        max_row_length = max(row_lengths)
        if max_row_length != len(headers):
            print(f"Skipping table {table_id+1} on page {page_num+1}. Row length mismatched with headers.")
            log_entries.append({
                "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "source_file": os.path.basename(pdf_path),
                "pdf_page": page_num + 1,
                "table_number": table_id + 1,
                "output_csv": "Not saved",
                "verdict": verdict,
                "issues": "; ".join(issues) if issues else "None",
            })        
            continue
        
        table_count += 1
#Transforms the information that GPT developed about each table_id into a dataframe that is then turned into a csv file. 
        df = pd.DataFrame(rows, columns = headers)
#Names the datafram that was just created, assigns it to a file path to be found in a certain folder, and officially turns the
#dataframe into a CSV file!
        filename = f"table_p{page_num+1}_t{table_id+1}.csv"
        filepath = os.path.join(outputdirectory, filename)
        df.to_csv(filepath, index=False)
#Now the CSV file that was just saved is loaded back into pandas to see if it saved correctly. 
#missing function counts all of the empty values in the table. If a table is deemed to be missing a value, then the "verdict"
# says "Requires Correction" (this prompts the user to look over the CSV and see if it is significantly different than the original table.
        df_check = pd.read_csv(filepath)
        missing = df_check.replace("", pd.NA).isna().sum().sum()
        if missing > 0:
            issues.append(f"{missing} missing or empty values found after re-read")
            verdict = "Requires Correction"
        print(f"\n Page {page_num + 1}, Table {table_id+1}: {verdict}")
 #The title is printed in the table based off of the csv if it can be parsed, to assign the verdict and issues to it.       
        if title:
            print:(f"Title: {title}")
# The issues for each of the tables is outputted in the log table, so the user knows what to go back and look at.    
        if issues:
            for issue in issues:
                print(f"Issues: {issue}")
        log_entries.append({
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "source_file": os.path.basename(pdf_path),
            "pdf_page": page_num + 1,
            "table_number": table_id + 1,
            "table_title": title if title else "Unknown",
            "output_csv": filename,
            "verdict": verdict,
            "issues": "; ".join(issues) if issues else "None",
            "suggestion": suggestion if suggestion else "None",
        })
            
pdf_document.close()

Opened PDF: C:\Users\tdmur\Downloads\The Scientific World Journal - 2025 - Addis - Intestinal Parasitic Infection  Prevalence and Associated Risk Factors at.pdf
Total Pages: 9

Processing page 1 of 9...
No tables found on page 1.

Processing page 2 of 9...
No tables found on page 2.

Processing page 3 of 9...
Tables found: 1
Sociodemographic characteristics of study participants, Delgi Primary Hospital, Central Gondar, Ethiopia (n = 404).

 Page 3, Table 1: Approved

Processing page 4 of 9...
Tables found: 1
Distribution of intestinal parasite species by sex and age group among study participants (n = 404).
Skipping table 1 on page 4. Row length mismatched with headers.

Processing page 5 of 9...
Tables found: 1
Bivariate logistic regression analysis of sociodemographic-related factors and IPIs among study subjects attending at Delgi Primary Hospital.

 Page 5, Table 1: Requires Correction
Issues: 19 missing or empty values found after re-read

Processing page 6 of 9...
Could not parse

In [7]:
#prints the log entries, creates the logentries csv in the same folder as the other tables and tells you if nothing was saved. 
if log_entries:
    log_df = pd.DataFrame(log_entries)
    log_path = os.path.join(outputdirectory, "extraction_log.csv")
    log_df.to_csv(log_path, index=False)
    print(f"Log saved to {log_path}\n")
    display(log_df)
else:
    print(" No tables were extracted. Log not saved.")

Log saved to data/outputs\extraction_log.csv



,timestamp,source_file,pdf_page,table_number,table_title,output_csv,verdict,issues,suggestion
0,2026-07-23 17:51:37,The Scientific World Journal - 2025 - Addis - ...,3,1,Sociodemographic characteristics of study part...,table_p3_t1.csv,Approved,None,None
1,2026-07-23 17:51:44,The Scientific World Journal - 2025 - Addis - ...,4,1,NaN,Not saved,Approved,None,NaN
2,2026-07-23 17:51:54,The Scientific World Journal - 2025 - Addis - ...,5,1,Bivariate logistic regression analysis of soci...,table_p5_t1.csv,Requires Correction,19 missing or empty values found after re-read,None
3,2026-07-23 17:52:11,The Scientific World Journal - 2025 - Addis - ...,7,1,NaN,Not saved,Approved,None,NaN


In [ ]:
Run Log
Date: 7/9-7/23
Task Performed: Created an AI agent that extracts tables in PDFs to CSV files, reviews each CSV file compared to the original PDF, and informs the user which tables should be reviewed and what the issues are.
Input filename: "C:\Users\tdmur\Downloads\The Scientific World Journal - 2025 - Addis - Intestinal Parasitic Infection  Prevalence and Associated Risk Factors at.pdf"
Notebook or script version: JupyterNotebook
Model: OpenAI GPT 4o Mini
Output filenames: extractionlog.csv, table_P3_t1.csv, table_P4_t1.csv, ..., etc.
Warnings: Does not work very well for tables with subheaders or renamed files. GPT handles things differently each time, more prompting needed.
Execution time: <2 minutes
API cost <$5
Final review/verification: not completely finished, version 1. 